In [10]:
import sys
from pathlib import Path
workspace_root = Path().cwd().parent.parent
print(f"Workspace root: {workspace_root}")
sys.path.insert(0, str(workspace_root))

Workspace root: e:\Project\news_secretary\apps\back


In [ ]:
import sqlite3
import json
import re

from app.db.insert_data import insert_keywords_record
from app.configuration import SQLITE_DB_PATH, KEYWORDS_FILE_PATH, TABLE_NAMES

In [12]:
raw = json.load(open("cptall-en-GB.json", "r", encoding="utf-8"))
raw = raw.get("conceptSet")
raw

[{'uri': 'http://cv.iptc.org/newscodes/mediatopic/01000000',
  'qcode': 'medtop:01000000',
  'type': ['http://www.w3.org/2004/02/skos/core#Concept'],
  'inScheme': ['http://cv.iptc.org/newscodes/mediatopic/'],
  'modified': '2021-02-18T12:00:00+00:00',
  'prefLabel': {'en-GB': 'arts, culture, entertainment and media'},
  'definition': {'en-GB': 'All forms of arts, entertainment, cultural heritage and media'},
  'narrower': ['medtop:20000002', 'medtop:20000038', 'medtop:20000045'],
  'exactMatch': ['http://cv.iptc.org/newscodes/subjectcode/01000000'],
  'created': '2009-10-22T02:00:00+00:00'},
 {'uri': 'http://cv.iptc.org/newscodes/mediatopic/02000000',
  'qcode': 'medtop:02000000',
  'type': ['http://www.w3.org/2004/02/skos/core#Concept'],
  'inScheme': ['http://cv.iptc.org/newscodes/mediatopic/'],
  'modified': '2021-05-05T12:00:00+00:00',
  'prefLabel': {'en-GB': 'crime, law and justice'},
  'definition': {'en-GB': 'The establishment and/or statement of the rules of behaviour in soci

In [13]:
pre_dict = {}
for item in raw:
    label = item.get("prefLabel").get('en-GB')
    qcode = item.get("qcode")
    definition = item.get("definition").get('en-GB')
    children = item.get("narrower", None)
    pre_dict[qcode] = {
        "label": label,
        "definition": definition,
        "synonyms": item.get("synonyms", [])
    }
    if children:
        pre_dict[qcode]["children"] = children

pre_dict

{'medtop:01000000': {'label': 'arts, culture, entertainment and media',
  'definition': 'All forms of arts, entertainment, cultural heritage and media',
  'synonyms': [],
  'children': ['medtop:20000002', 'medtop:20000038', 'medtop:20000045']},
 'medtop:02000000': {'label': 'crime, law and justice',
  'definition': 'The establishment and/or statement of the rules of behaviour in society, the enforcement of these rules, breaches of the rules, the punishment of offenders and the organisations and bodies involved in these activities',
  'synonyms': [],
  'children': ['medtop:20000082',
   'medtop:20000106',
   'medtop:20000119',
   'medtop:20000121',
   'medtop:20000129']},
 'medtop:03000000': {'label': 'disaster, accident and emergency incident',
  'definition': 'Man made or natural event resulting in loss of life or injury to living creatures and/or damage to inanimate objects or property',
  'synonyms': [],
  'children': ['medtop:20000139',
   'medtop:20000148',
   'medtop:20000160',
 

In [14]:
organized_dict = {}
for qcode, item in pre_dict.items():
    organized_dict[qcode] = {
        "label": item["label"],
        "definition": item["definition"],
        "synonyms": item.get("synonyms", None),
        "parent": None,
        "children": item.get("children", None),
    }

for qcode, item in pre_dict.items():
    if item.get('children'):
        for child_code in item['children']:
            organized_dict[child_code]["parent"] = qcode

organized_dict

{'medtop:01000000': {'label': 'arts, culture, entertainment and media',
  'definition': 'All forms of arts, entertainment, cultural heritage and media',
  'synonyms': [],
  'parent': None,
  'children': ['medtop:20000002', 'medtop:20000038', 'medtop:20000045']},
 'medtop:02000000': {'label': 'crime, law and justice',
  'definition': 'The establishment and/or statement of the rules of behaviour in society, the enforcement of these rules, breaches of the rules, the punishment of offenders and the organisations and bodies involved in these activities',
  'synonyms': [],
  'parent': None,
  'children': ['medtop:20000082',
   'medtop:20000106',
   'medtop:20000119',
   'medtop:20000121',
   'medtop:20000129']},
 'medtop:03000000': {'label': 'disaster, accident and emergency incident',
  'definition': 'Man made or natural event resulting in loss of life or injury to living creatures and/or damage to inanimate objects or property',
  'synonyms': [],
  'parent': None,
  'children': ['medtop:20

In [15]:
len(organized_dict)

1393

In [16]:
# load added synonyms keywords file
added_synonyms = json.load(open("iptc_media_topics_with_synonyms.json", "r", encoding="utf-8"))

len(added_synonyms)

1393

In [ ]:
# save to database
with sqlite3.connect(SQLITE_DB_PATH) as conn:
    with conn:
        cursor = conn.cursor(row_factory=dict_row)

        for id , item in added_synonyms.items():
            insert_keywords_record(item, cursor)

In [18]:
conn.close()